# 05 - Exemplo de RAG (Retrieval-Augmented Generation) 

Neste notebook, vamos demonstrar como utilizar o modelo `ricardoz/BERTugues-base-portuguese-cased` como o motor de **Busca Semântica (Retrieval)** em uma arquitetura de RAG.

O RAG é composto por duas etapas principais:
1. **Retrieval (Recuperação)**: Dada uma pergunta (query), buscamos em uma base de conhecimento os textos mais relevantes. É aqui que o **BERTugues** brilha! Ele transforma os textos em vetores (*embeddings*), e nós usamos a distância entre esses vetores para achar a informação exata.
2. **Generation (Geração)**: O texto recuperado é passado como "contexto" para um modelo de linguagem generativo (como GPT, LLaMA, Gemini etc.) para que ele escreva a resposta final baseada apenas naqueles fatos.

Neste exemplo, focaremos em construir a primeira etapa completa: um **Retriever Denso** 100% em português usando o seu modelo.

## 1. Instalando as bibliotecas necessárias
Vamos usar o `scikit-learn` para calcular a Similaridade de Cosseno (Cosine Similarity) entre os embeddings.

In [1]:
# !pip install transformers torch scikit-learn numpy

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

## 2. Carregando o Tokenizador e o Modelo
Usaremos o modelo base para extrair os *hidden states* das palavras.

In [ ]:
model_name = "ricardoz/BERTugues-base-portuguese-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval(); # Modo de inferência

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ricardoz/BERTugues-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

## 3. Função para Gerar Embeddings de Sentenças (*Sentence Embeddings*)
Para representar uma frase inteira, uma técnica muito comum (usada no Sentence-BERT) é o **Mean Pooling**: tiramos a média dos embeddings de todos os tokens da frase, ignorando os tokens de preenchimento (padding).

In [3]:
def get_sentence_embedding(text):
    # Tokenizar o texto
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Pegar todos os hidden states recebidos da última camada:
    # shape: (batch_size, sequence_length, hidden_size)
    token_embeddings = outputs.last_hidden_state
    attention_mask = inputs['attention_mask']
    
    # === Mean Pooling ===
    # Precisamos expandir a máscara de atenção para ter a mesma dimensão do token_embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    
    # Soma dos embeddings ponderada pela máscara (zeramos os paddings)
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    
    # Contagem de tokens reais para poder dividir e fazer a média
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    
    # Embedding médio final da frase
    mean_pooled = sum_embeddings / sum_mask
    
    return mean_pooled.cpu().numpy()[0] # retornar como array 1D numpy

## 4. Criando a Base de Conhecimento (Knowledge Base)
No mundo real, isso seria uma coleção gigante de PDFs, páginas da web, wikis internas, divididas em "chunks" (blocos) de texto. 
Aqui, criaremos uma lista de textos simples com algumas informações específicas.

In [4]:
base_conhecimento = [
    "A capital da França é Paris, famosa pela Torre Eiffel e pelo museu do Louvre.",
    "O modelo BERTugues foi treinado para compreender profundamente a língua portuguesa usando técnicas modernas.",
    "Para fazer um bolo de cenoura, você precisa de cenouras, farinha, açúcar, ovos e óleo.",
    "Em 1969, a missão Apollo 11 levou os primeiros seres humanos à superfície da Lua.",
    "A respiração celular é o processo pelo qual as células produzem energia a partir da glicose.",
    "A biblioteca Transformers da Hugging Face facilita o uso de LLMs e modelos de NLP na prática."
]

print("Indexando (Vetorizando) a Base de Conhecimento...")
# Geramos uma matriz contendo o embedding para cada documento da nossa base
doc_embeddings = np.array([get_sentence_embedding(doc) for doc in base_conhecimento])
print(f"Foram indexados {doc_embeddings.shape[0]} documentos com dimensão {doc_embeddings.shape[1]}.")

Indexando (Vetorizando) a Base de Conhecimento...
Foram indexados 6 documentos com dimensão 768.


## 5. O Processo de Busca (Retrieval)
Agora iremos receber uma pergunta do usuário. Transformaremos essa pergunta também em um vetor e mediremos a **Similaridade de Cosseno** (Cosine Similarity) entre esse vetor e todos os vetores da nossa Base de Conhecimento. Os documentos com maior pontuação são os mais semanticamente próximos da pergunta.

In [5]:
def retriever_search(query, top_k=2):
    # 1. Geramos o embedding para a pergunta
    query_emb = get_sentence_embedding(query)
    
    # 2. Calculamos a similaridade de cosseno contra toda a Base (1xN contra Nx768)
    # A função cosine_similarity espera matrizes 2D, então fazemos query_emb.reshape(1, -1)
    similaridades = cosine_similarity(query_emb.reshape(1, -1), doc_embeddings)[0]
    
    # 3. Pega os índices dos maiores valores de similaridade
    # argsort retorna de ordem crescente, então pegamos do final pro começo ([::-1])
    top_indices = np.argsort(similaridades)[::-1][:top_k]
    
    resultados = []
    for idx in top_indices:
        resultados.append({
            "texto": base_conhecimento[idx],
            "score": similaridades[idx]
        })
        
    return resultados

## 6. Testando a RAG / Busca Semântica
Vamos fazer uma pergunta e ver qual documento o BERTugues consegue resgatar!

In [6]:
perguntas_teste = [
    "Que ingredientes eu uso para fazer uma sobremesa de cenoura?",
    "Como as células conseguem energia?",
    "Qual a função do modelo BERT em português?"
]

for p in perguntas_teste:
    print(f"\n❓ Pergunta: '{p}'")
    contextos = retriever_search(p, top_k=1) # Queremos apenas o 1º documento mais relevante
    
    print(f"✅ Documento Recuperado (Score: {contextos[0]['score']:.4f}):")
    print(f"   {contextos[0]['texto']}")


❓ Pergunta: 'Que ingredientes eu uso para fazer uma sobremesa de cenoura?'
✅ Documento Recuperado (Score: 0.7390):
   Para fazer um bolo de cenoura, você precisa de cenouras, farinha, açúcar, ovos e óleo.

❓ Pergunta: 'Como as células conseguem energia?'
✅ Documento Recuperado (Score: 0.6836):
   A respiração celular é o processo pelo qual as células produzem energia a partir da glicose.

❓ Pergunta: 'Qual a função do modelo BERT em português?'
✅ Documento Recuperado (Score: 0.7114):
   A biblioteca Transformers da Hugging Face facilita o uso de LLMs e modelos de NLP na prática.


## 7. A última etapa (Apenas ilustrativa)
Nos sistemas completos de RAG, o documento recuperado e a pergunta seriam passados em um prompt (ex: para o ChatGPT, Claude ou LLamas locais) com a seguinte instrução:

```
Responda a pergunta do usuário usando APENAS o contexto fornecido abaixo.

Contexto:
Para fazer um bolo de cenoura, você precisa de cenouras, farinha, açúcar, ovos e óleo.

Pergunta:
Que ingredientes eu uso para fazer uma sobremesa de cenoura?
```

E o LLM responderia: *"Você precisará de cenouras, farinha, açúcar, ovos e óleo."*

E pronto! Evitamos alucinações injetando o contexto real – contexto esse que foi achado usando ótimos **embeddings do BERTugues** e busca semântica de precisão!